In [1]:
import os 
import sys 
import numpy as np 
import torch 
import matplotlib.pyplot as plt 
from tqdm import tqdm 
import time 
import random 
from datetime import datetime 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

from src.dataset.custom_dataset import GenDataset, DiscDataset
from src.model.graph_model import NEGATGenerator, DiffPoolDiscriminator
from src.training.trainer import train_GAN

from utils.gen_utils import load_config, get_device, dataset_splitter, generate_markdown_report_GAN_and_save_model
from utils.ppnet_utils import initialize_network
from utils.load_data_utils import load_sampled_input_data

yaml_config= load_config('config_gan.yaml')

device = get_device(yaml_config['device'])

net = initialize_network(net_name=yaml_config['data']['net_name'],
                        #  net_name=yaml_config['data']['net_name'], 
                         load_std=yaml_config['data']['load_std']) 

########### data for generator 
sampled_input_data_G = load_sampled_input_data(sc_type=yaml_config['data']['gen_scenario_type'], 
                                               net=net, 
                                               num_samples=yaml_config['data']['num_samples'], 
                                               noise=yaml_config['data']['noise'])

# sparsity of the node and edge features 
node_feat_sparsity = np.count_nonzero(sampled_input_data_G['node_mask']) / sampled_input_data_G['node_mask'].numpy().size
pflow_edge_sparsity = np.count_nonzero(sampled_input_data_G['edge_mask'][:,:,0]) / sampled_input_data_G['edge_mask'][:,:,0].numpy().size

print(f"Sparsity of PV measurements at buses = {node_feat_sparsity:.1f}%")
print(f"Sparsity of P+ measurements at branches = {pflow_edge_sparsity:.1f}")

########### data for discriminator 
sampled_input_data_D = load_sampled_input_data(sc_type=yaml_config['data']['dis_scenario_type'], 
                                               net=net, 
                                               num_samples=yaml_config['data']['num_samples'], 
                                               noise=yaml_config['data']['noise'])

dataset_G = GenDataset(model_name=yaml_config['model_G']['name'], 
                       sampled_input_data=sampled_input_data_G)

(train_loader_G, val_loader_G, test_loader_G), _ = dataset_splitter(dataset_G, 
                                                                    batch_size=yaml_config['loader']['batch_size'])

dataset_D = DiscDataset(sampled_input_data=sampled_input_data_D)

(train_loader_D, val_loader_D, test_loader_D), _ = dataset_splitter(dataset_D,
                                                                    batch_size=yaml_config['loader']['batch_size'])

###########################################################
seeds = np.arange(100)


all_losses_seeds = {}
generated_data = {}
simulated_v_pf_data = {}
all_time_counters = {}

for seed in tqdm(seeds):
    
    random.seed(int(seed))
    np.random.seed(seed)
    torch.manual_seed(seed)

    start_time_training = time.perf_counter()
    # instantiate model, optimizer and schedular for Generator 
    model_G = NEGATGenerator(node_input_features=dataset_G[0][0].x.shape[-1], 
                        list_node_hidden_features=yaml_config['model_G']['list_node_hidden_features'], # [128,64], 
                        node_out_features=yaml_config['model_G']['node_out_features'], # 64, 
                        k_hop_node=yaml_config['model_G']['k_hop_node'], #1, 
                        edge_input_features=dataset_G[0][1].x.shape[-1], 
                        list_edge_hidden_features=yaml_config['model_G']['list_edge_hidden_features'], #[128,64], 
                        edge_output_features=yaml_config['model_G']['edge_out_features'], #64, 
                        k_hop_edge=yaml_config['model_G']['k_hop_edge'], #1, 
                        gat_out_features=yaml_config['model_G']['gat_out_features'], #32, 
                        gat_head=yaml_config['model_G']['gat_head'], #2, 
                        device=device)

    optimizer_G = torch.optim.Adam(model_G.parameters(), 
                                lr=yaml_config['training_G']['lr'], 
                                weight_decay=yaml_config['training_G']['weight_decay'])

    schedular_G = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_G, 
                                                        mode='min', 
                                                        factor=0.1, 
                                                        min_lr=yaml_config['training_G']['schedular_min_lr'])

    total_params_G = sum(p.numel() for p in model_G.parameters() if p.requires_grad)
    print(f'Total number of parameters of model {model_G}: {total_params_G}')

    # instantiate model, optimizer and schedular for Discriminator 
    model_D = DiffPoolDiscriminator(in_channel=dataset_D[0].x.shape[-1], 
                                hidden_channel=yaml_config['model_D']['hidden_channel'], 
                                out_channel=yaml_config['model_D']['out_channel'], 
                                num_nodes=len(net.bus.index))

    total_params_D = sum(p.numel() for p in model_D.parameters() if p.requires_grad)
    print(f'Total number of parameters of model {model_D}: {total_params_D}')

    optimizer_D = torch.optim.Adam(model_D.parameters(), 
                                lr=yaml_config['training_D']['lr'], 
                                weight_decay=yaml_config['training_D']['weight_decay'])

    schedular_D = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_D, 
                                                        mode='min', 
                                                        factor=0.1, 
                                                        min_lr=yaml_config['training_D']['schedular_min_lr'])
    
    all_losses_seeds[seed] = train_GAN(model_G=model_G, 
                                        model_D=model_D, 
                                        all_loader_G= [train_loader_G, val_loader_G, test_loader_G], 
                                        all_loader_D= [train_loader_D, val_loader_D, test_loader_D],  
                                        optimizer_G=optimizer_G, 
                                        optimizer_D=optimizer_D, 
                                        schedular_G=schedular_G, 
                                        schedular_D=schedular_D, 
                                        num_epoch=yaml_config['training_GAN']['num_epoch'], 
                                        disc_iter=yaml_config['training_GAN']['disc_iter'], 
                                        gen_iter=yaml_config['training_GAN']['gen_iter'],
                                        feature_matching=yaml_config['training_GAN']['feature_matching'],   
                                        device=device)

    current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_dir_gan = f"{parent_dir}/results/GAN_only/{current_time}_{seed}"
    generated_data[seed], simulated_v_pf_data[seed] = generate_markdown_report_GAN_and_save_model(report_dir=report_dir_gan,
                                                                    yaml_config=yaml_config, 
                                                                    train_g_losses=all_losses_seeds[seed]['train_g_losses'], 
                                                                    train_d_losses=all_losses_seeds[seed]['train_d_losses'], 
                                                                    train_d_accuracies=all_losses_seeds[seed]['train_d_accuracies'], 
                                                                    test_loader_G=test_loader_G, 
                                                                    model_G=model_G, 
                                                                    sampled_input_data_G=sampled_input_data_G, 
                                                                    return_data=True)
    
    end_time_training = time.perf_counter() 

    all_time_counters[seed] = end_time_training - start_time_training


NameError: 
 Invalid Network Name! 

In [9]:
import joblib 
joblib.dump(generated_data, parent_dir + f'/results/GAN_only/Oct3_all_seed_generated_data.pkl')
joblib.dump(all_losses_seeds, parent_dir + f'/results/GAN_only/Oct3_all_seed_losses_data.pkl')
joblib.dump(simulated_v_pf_data, parent_dir + f'/results/GAN_only/Oct3__all_seed_simulated_v_pf_data.pkl')
joblib.dump(all_time_counters, parent_dir + f'/results/GAN_only/Oct3_all_time_counters.pkl')

['/Users/sohamprajapati/Documents/Thesis/Project Codes/TapSEGNN/results/GAN_only/Oct3_all_time_counters.pkl']

In [10]:
import pandas as pd 
# plot the variance of simulated vs. generated voltages 
generated_v_df = pd.DataFrame(columns=seeds)
simulated_v_pf_data_df = pd.DataFrame(columns=seeds)
all_val_g_losses_df = pd.DataFrame(columns=seeds)
all_val_d_losses_df = pd.DataFrame(columns=seeds)
all_val_d_accuracies_df = pd.DataFrame(columns=seeds)

num_buses = len(net.bus)

for seed in seeds: 
    all_val_g_losses_df[seed] = all_losses_seeds[seed]['val_g_losses']
    all_val_d_losses_df[seed] = all_losses_seeds[seed]['val_d_losses']
    all_val_d_accuracies_df[seed] = all_losses_seeds[seed]['val_d_accuracies']
    try: 
        generated_v_df[seed] = generated_data[seed]['gen_v'][:num_buses]
        simulated_v_pf_data_df[seed] = simulated_v_pf_data[seed]
    except Exception as e: 
        print(e)
generated_v_df.dropna(axis='columns', inplace=True)
simulated_v_pf_data_df.dropna(axis='columns', inplace=True)
# simulated_v_pf_data_df, generated_v_df

In [6]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Calculate mean and standard deviation for both dataframes
simulated_mean = simulated_v_pf_data_df.mean(axis=1)
simulated_std = simulated_v_pf_data_df.std(axis=1)
generated_mean = generated_v_df.mean(axis=1)
generated_std = generated_v_df.std(axis=1)

# Create the plot
# plt.figure(figsize=(10, 6))
# x = range(len(simulated_mean))
# sns.lineplot(x=x, y=simulated_mean, label='Simulated')
# plt.fill_between(x, simulated_mean - simulated_std, simulated_mean + simulated_std, alpha=0.3)
# sns.lineplot(x=x, y=generated_mean, label='Generated')
# plt.fill_between(x, generated_mean - generated_std, generated_mean + generated_std, alpha=0.3)
# plt.xlabel('Bus')
# plt.ylabel('Voltage [p.u]')
# plt.legend()
# plt.show()

NameError: name 'simulated_v_pf_data_df' is not defined

In [ ]:
val_d_losses_mean = all_val_d_losses_df.mean(axis=1)
val_d_losses_std = all_val_d_losses_df.std(axis=1)

val_g_losses_mean = all_val_g_losses_df.mean(axis=1)
val_g_losses_std = all_val_g_losses_df.std(axis=1)

val_d_accuracies_mean = all_val_d_accuracies_df.mean(axis=1)
val_d_accuracies_std = all_val_d_accuracies_df.std(axis=1)

fig, ax = plt.subplots(3,1, figsize=(8,10))
x = range(len(val_d_losses_mean))

sns.lineplot(x=x, y=val_d_losses_mean, ax=ax[0], label='D Loss')
ax[0].fill_between(x, val_d_losses_mean - val_d_losses_std, val_d_losses_mean + val_d_losses_std, alpha=0.3)

sns.lineplot(x=x, y=val_g_losses_mean, ax=ax[1], label='G Loss')
ax[1].fill_between(x, val_g_losses_mean - val_g_losses_std, val_g_losses_mean + val_g_losses_std, alpha=0.3)

sns.lineplot(x=x, y=val_d_accuracies_mean, ax=ax[2], label='D Accuracy')
ax[2].fill_between(x, val_d_accuracies_mean - val_d_accuracies_std, val_d_accuracies_mean + val_d_accuracies_std, alpha=0.3)

plt.tight_layout()
plt.show()

In [11]:
# check which run has the least difference between simulated and generated 
gen_minus_sim = generated_v_df - simulated_v_pf_data_df
np.argmin(gen_minus_sim.abs().sum(axis=0))

np.int64(98)

In [12]:
np.min(gen_minus_sim.abs().sum(axis=0))

np.float64(0.4247866107774039)

In [13]:
gen_minus_sim.abs().sum(axis=0)

0     0.613224
1     0.554295
2     0.784649
3     0.703526
4     0.838163
        ...   
95    0.719680
96    0.836055
97    0.461966
98    0.424787
99    0.566119
Length: 100, dtype: float64